[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IshtVibhu/Python-for-Computational-Economics/blob/main/NB_07_QuantEcon.ipynb)

# Python for Computational Economics
## Notebook 07 — QuantEcon: Dynamic Programming and Macroeconomics

**Prerequisites:** NB_01–NB_05

**What you will learn:**
- Markov chains: stochastic business cycles, long-run distributions
- Value function iteration: the consumer's saving problem
- Job search model: McCall's sequential search
- Asset pricing: Lucas tree model

`quantecon` was created by Thomas Sargent and John Stachurski to accompany their graduate macroeconomics textbooks at [quantecon.org](https://quantecon.org).

In [ ]:
try:
    import quantecon as qe
    import numpy as np
    import matplotlib.pyplot as plt
    import pandas as pd
except ImportError:
    !pip install quantecon numpy matplotlib pandas --quiet
    import quantecon as qe
    import numpy as np
    import matplotlib.pyplot as plt
    import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
print(f"QuantEcon {qe.__version__}")

---
## 1. Markov Chains — Business Cycle Dynamics

A **Markov chain** is a probabilistic model where the probability of being in state `j` tomorrow depends only on the current state `i`. Business cycle models often use Markov chains to model regime switching between expansion and recession.

In [ ]:
# Two-state business cycle Markov chain
# States: 0 = Recession, 1 = Expansion
# Transition matrix P[i,j] = probability of moving from state i to state j
P = np.array([
    [0.70, 0.30],   # From Recession: 70% stay, 30% recover
    [0.05, 0.95],   # From Expansion: 5% contract, 95% stay
])

mc = qe.MarkovChain(P, state_values=["Recession", "Expansion"])

# Stationary distribution
stationary = mc.stationary_distributions[0]
print("Stationary distribution (long-run fractions of time):")
for state, prob in zip(["Recession", "Expansion"], stationary):
    print(f"  {state}: {prob:.3f} ({prob*100:.1f}% of time)")

# Simulate 200 quarters
np.random.seed(42)
path = mc.simulate(ts_length=200, init=1)  # start in expansion

recession_quarters = np.sum(path == 0)
print(f"\nSimulated 200 quarters:")
print(f"  Quarters in Recession: {recession_quarters} ({recession_quarters/200:.1%})")
print(f"  Quarters in Expansion: {200-recession_quarters} ({(200-recession_quarters)/200:.1%})")

# Mean first passage times
print(f"\nMean return time to Recession: {mc.mean_first_passage_times()[1, 0]:.1f} quarters")
print(f"Mean return time to Expansion: {mc.mean_first_passage_times()[0, 1]:.1f} quarters")

In [ ]:
# Map business cycle states to GDP growth
gdp_given_state = {0: -2.0, 1: 3.0}  # Recession: -2%, Expansion: +3%
gdp_sim = [gdp_given_state[s] + np.random.normal(0, 0.5) for s in path]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

ax1.fill_between(range(200), path, alpha=0.4, color="steelblue", label="Expansion (1)")
ax1.set_ylabel("State"); ax1.set_yticks([0, 1]); ax1.set_yticklabels(["Recession", "Expansion"])
ax1.set_title("Simulated Business Cycle (Markov Chain)")

ax2.bar(range(200), gdp_sim, color=["#d62728" if g < 0 else "#1f77b4" for g in gdp_sim], width=1)
ax2.axhline(0, color="black", linewidth=0.8)
ax2.set_xlabel("Quarter"); ax2.set_ylabel("GDP Growth (%)")

plt.tight_layout()
plt.savefig("business_cycle_markov.png", dpi=100, bbox_inches="tight")
plt.show()

---
## 2. Value Function Iteration — Consumption-Savings Problem

An infinitely-lived consumer with log utility chooses savings `s` in each period:

`V(w) = max_{0 ≤ s ≤ w} { log(w - s) + β * V(R*s) }`

We solve this by iterating on the Bellman equation until the value function converges.

In [ ]:
# Parameters
beta  = 0.96   # discount factor
R     = 1.04   # gross return on savings
n_w   = 200    # wealth grid points
w_max = 10.0
w_grid = np.linspace(0.01, w_max, n_w)

# Analytical solution for log utility: savings rate = beta * R (constant fraction)
# V*(w) = A + B*log(w) where B = 1/(1-beta), A = (beta*log(beta*R) + log(1-beta)) / (1-beta)
savings_rate_true = beta  # optimal savings rate

# Value function iteration
V = np.zeros(n_w)  # initial guess: V = 0

for iteration in range(500):
    V_new = np.empty(n_w)
    for i, w in enumerate(w_grid):
        # Possible savings: 0 to w (stay on grid)
        savings = w_grid[w_grid <= w]
        if len(savings) == 0:
            V_new[i] = -1e10
            continue
        consumption = w - savings
        R_savings   = np.minimum(R * savings, w_max)  # clamp to grid
        # Interpolate V at R*savings
        V_next = np.interp(R_savings, w_grid, V)
        bellman = np.log(np.maximum(consumption, 1e-8)) + beta * V_next
        V_new[i] = bellman.max()
    if np.max(np.abs(V_new - V)) < 1e-6:
        print(f"Converged after {iteration+1} iterations")
        break
    V = V_new

# Policy function: optimal savings at each wealth level
policy_savings = np.empty(n_w)
for i, w in enumerate(w_grid):
    savings = w_grid[w_grid <= w]
    if len(savings) == 0:
        policy_savings[i] = 0
        continue
    consumption = w - savings
    R_savings = np.minimum(R * savings, w_max)
    V_next = np.interp(R_savings, w_grid, V)
    bellman = np.log(np.maximum(consumption, 1e-8)) + beta * V_next
    policy_savings[i] = savings[np.argmax(bellman)]

print(f"\nAt w=5: optimal savings = {np.interp(5.0, w_grid, policy_savings):.3f}")
print(f"Analytical savings rate * w = {savings_rate_true * 5.0:.3f}")

---
## 3. Job Search — McCall's Sequential Search Model

An unemployed worker receives wage offers `w` drawn i.i.d. from distribution `F`. The worker accepts if `w ≥ w*` (reservation wage) and continues searching otherwise.

The reservation wage satisfies:
`w* = (1-β) * c + β * E[max(w*, w)]`

In [ ]:
from quantecon.models import McCallModel

# Parameters
mc_model = McCallModel(
    alpha=0.1,    # job destruction rate (per period)
    beta=0.96,    # discount factor
    c=6.0,        # unemployment benefit
    sigma=1.0,    # utility curvature
    w_min=10.0,
    w_max=60.0,
    w_size=50,
)

V, U = mc_model.solve_mccall_model()

# Reservation wage: lowest w where V[i] >= U
w_grid = np.linspace(10, 60, 50)
above_reservation = w_grid[V >= U]
w_star = above_reservation[0] if len(above_reservation) > 0 else w_grid[-1]

print(f"Reservation wage w* ≈ {w_star:.1f}")
print(f"Worker accepts offers ≥ {w_star:.1f}")
print(f"Acceptance probability: {(w_grid >= w_star).mean():.1%} of offers")

# Effect of unemployment benefit on reservation wage
print("\nEffect of unemployment benefit on reservation wage:")
for c in [4, 6, 8, 10, 12]:
    mc_c = McCallModel(alpha=0.1, beta=0.96, c=c, sigma=1.0, w_min=10, w_max=60, w_size=50)
    V_c, U_c = mc_c.solve_mccall_model()
    above = w_grid[V_c >= U_c]
    wr = above[0] if len(above) > 0 else 60
    print(f"  c = {c:2d}  →  w* ≈ {wr:.1f}")

---
## Your Turn — Hamilton Regime-Switching Chain

Hamilton (1989) modelled US GDP growth as switching between two regimes with:

```
P = [[0.9, 0.1],   # from Recession: 90% stay, 10% exit
     [0.2, 0.8]]   # from Expansion: 20% enter recession, 80% stay
```

1. Create a `MarkovChain` with this matrix
2. Compute the stationary distribution
3. Simulate 100 periods and count recession quarters
4. Report the expected duration of a recession episode (hint: for an absorbing state with self-transition `p`, expected duration = `1/(1-p)`)

In [ ]:
# Your code here


In [ ]:
#@title Solution
P_hamilton = np.array([[0.9, 0.1], [0.2, 0.8]])
mc_h = qe.MarkovChain(P_hamilton, state_values=["Recession", "Expansion"])

stat = mc_h.stationary_distributions[0]
print("Hamilton (1989) Markov Chain:")
print(f"  Stationary: Recession = {stat[0]:.3f}, Expansion = {stat[1]:.3f}")

np.random.seed(10)
path_h = mc_h.simulate(ts_length=100, init=1)
n_rec  = np.sum(path_h == 0)
print(f"  Recession quarters (100 periods): {n_rec}")

p_stay_rec = P_hamilton[0, 0]
expected_duration = 1 / (1 - p_stay_rec)
print(f"  Expected recession duration: {expected_duration:.1f} quarters")

---
## Summary

| Concept | QuantEcon tool |
|---|---|
| Markov chain | `qe.MarkovChain(P)` |
| Stationary distribution | `.stationary_distributions[0]` |
| Simulate path | `.simulate(ts_length=T)` |
| Job search | `qe.models.McCallModel(...)` |
| Value function iteration | Custom loop + `np.interp` |

**Reference:** Sargent, T. & Stachurski, J. *Quantitative Economics with Python*. quantecon.org

**Next:** NB_08_Mesa_ABM.ipynb — building agent-based models.